# TCM Embedding Space (TCM-ES): Demonstration Workflow

This notebook provides a block-by-block demonstration of the analysis workflow for:

**_An Interpretable AI Framework Quantifying TCM Principles towards Integration with Modern Biomedicine_**

The core TCM-ES model is a Transformer-based autoencoder trained on matched symptom patterns and prescribed herbal formulas. The model combines bottleneck encoding, self- and cross-attention, reconstruction, and contrastive learning to represent symptom patterns and formulas in a shared 256-dimensional embedding space.

Core TCM-ES training uses only matched symptom-pattern–formula records. Differentiated syndrome labels, herb-property annotations, clinical outcomes, and biomedical information are not used as model inputs or training targets. These independent annotations and datasets are used only in downstream interpretation, validation, and biomedical mapping.

> ## Important note on the public example data
>
> **This notebook uses public example datasets to demonstrate code execution and the TCM-ES analysis workflow. Numerical results obtained from these example subsets are not expected to reproduce the exact results reported in the manuscript, which were obtained from the full study datasets.**
>
> Raw patient-level clinical datasets are not publicly deposited because of privacy and data-use restrictions. After publication, de-identified clinical data may be requested from the corresponding author and may be made available upon reasonable request, subject to applicable data-use agreements and institutional requirements.
>
> The released pretrained checkpoints are recommended for reproducing downstream workflows. Training from scratch is optional and substantially more computationally demanding.

## Notebook organization

The notebook follows the analysis sequence used in the repository:

1. Prepare model inputs, splits, and benchmark data.
2. Train or load the core TCM-ES models and generate embeddings.
3. Characterize TCM-ES geometry, model attention, and conventional TCM principle-associated organization.
4. Evaluate reproducibility across independently trained models.
5. Evaluate formula prediction and retrospective clinical relationships.
6. Map biomedical entities into the fixed TCM-ES and examine PPI-network concordance.

Run the notebook from the **repository root directory** so that relative paths such as `bin/`, `core/`, `data/`, and `results/` resolve correctly.

## 0. Environment and imports

In [ ]:
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import pickle
import random
import subprocess
import sys
from pathlib import Path

import pandas as pd
import torch

from bin.TCM_embedding_generator import TCMEmbeddingGenerator

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")
print("Working directory:", Path.cwd())

## 1. Prepare data for model training and evaluation

The primary TCM-ES model is trained using matched symptom-pattern–formula records. This section prepares the train/validation/test split and model dataloaders used by the core model, as well as the processed external TCM-PD benchmark data.

The public demonstration uses `TCM_formula_data_example_2000.pkl`; therefore, results from this notebook are demonstration results rather than manuscript-scale estimates.

### 1.1 Primary train/validation/test split and dataloaders

In [ ]:
subprocess.run(
    [
        sys.executable,
        "bin/data_generator.py",
        "data/training",
        "--formula-data",
        "data/TCM_formulas/TCM_formula_data_example_2000.pkl",
        "--make-dataloader",
    ],
    check=True,
)

### 1.2 Repeated splits for training-seed reproducibility analyses

In [ ]:
subprocess.run(
    [
        sys.executable,
        "bin/data_generator.py",
        "data/training",
        "--formula-data",
        "data/TCM_formulas/TCM_formula_data_example_2000.pkl",
        "--num-repeats",
        "10",
    ],
    check=True,
)

### 1.3 Prepare the external TCM-PD benchmark dataset

In [ ]:
subprocess.run(
    [
        sys.executable,
        "bin/data_generator.py",
        "--make-external-pd-all-valid",
        "--external-xlsx",
        "data/TCM_PD_external/TCM_PD_combined_30527.xlsx",
        "--external-out-dir",
        "data/TCM_PD_external",
    ],
    check=True,
)

## 2. Core TCM-ES model training

The core model uses separate symptom-pattern and formula encoders and decoders. Matched symptom-pattern–formula pairs are aligned within a shared bottleneck space through contrastive learning, while reconstruction objectives preserve the information required for bidirectional symptom-pattern/formula inference.

### Pretrained checkpoints

For most users, training from scratch is unnecessary. Download the released checkpoints from the repository **Releases** page and place them under:

```text
core/trained_model/model_epoch_60.pkl

core/trained_model_repeated/
├── repeat_01/model_epoch_*.pkl
├── repeat_02/model_epoch_*.pkl
├── ...
└── repeat_10/model_epoch_*.pkl
```

The optimal checkpoint epoch may differ across the ten independently trained repeat models.

The two cells below are therefore **optional** and should only be run when retraining the models from scratch.

### 2.1 Optional: train the primary TCM-ES model

In [ ]:
data_dir = "data/training"
save_dir = "core/trained_model"
show_loss = "N"
beta = 1.5
train_mode = "full"

subprocess.run(
    [
        sys.executable,
        "bin/train_main_model.py",
        data_dir,
        save_dir,
        show_loss,
        str(beta),
        train_mode,
    ],
    check=True,
)

After training, select the checkpoint according to the validation criterion used by the model-training workflow. The released primary checkpoint used for downstream analyses is `model_epoch_60.pkl`.

### 2.2 Optional: independently train the repeat models

In [ ]:
data_dir = "data/training"
repeat_save_base_dir = "core/trained_model_repeated"
show_loss = "N"
beta = 1.5
train_mode = "full"
num_repeats = 10

subprocess.run(
    [
        sys.executable,
        "bin/data_generator.py",
        "data/training",
        "--formula-data",
        "data/TCM_formulas/TCM_formula_data_example_2000.pkl",
        "--num-repeats",
        str(num_repeats),
        "--make-dataloader",
    ],
    check=True,
)

for repeat_id in range(1, num_repeats + 1):
    repeat_data_dir = os.path.join(
        data_dir, "repeated_splits", f"repeat_{repeat_id:02d}"
    )
    repeat_save_dir = os.path.join(
        repeat_save_base_dir, f"repeat_{repeat_id:02d}"
    )

    subprocess.run(
        [
            sys.executable,
            "bin/train_main_model.py",
            repeat_data_dir,
            repeat_save_dir,
            show_loss,
            str(beta),
            train_mode,
        ],
        check=True,
    )

    for fname in [
        "train_dataloader.pkl",
        "val_dataloader.pkl",
        "test_dataloader.pkl",
    ]:
        fpath = os.path.join(repeat_data_dir, fname)
        if os.path.exists(fpath):
            os.remove(fpath)

## 3. Generate TCM-ES embeddings

After training, the fixed encoders map standardized symptoms, herbs, complete symptom patterns, and formulas into the shared TCM-ES.

For the public demonstration, the primary checkpoint and example formula records are used below.

### 3.1 Generate embeddings and held-out cross-attention using the primary model

In [ ]:
subprocess.run(
    [
        sys.executable,
        "bin/TCM_embedding_generator.py",
        "--model-dir",
        "core/trained_model/model_epoch_60.pkl",
        "--tasks",
        "formula",
        "individual",
        "external-tcm-pd",
        "general-clinical",
        "attention",
    ],
    check=True,
)

The `attention` task extracts bidirectional cross-attention from matched held-out symptom-pattern–formula records. These saved attention maps are used later for the attention-guided token-removal perturbation analysis.

### 3.2 Generate formula and individual-entity embeddings for the ten repeat models

In [ ]:
repeat_model_root = Path("core/trained_model_repeated")

for repeat_id in range(1, 11):
    repeat_name = f"repeat_{repeat_id:02d}"
    model_dir = repeat_model_root / repeat_name
    model_files = sorted(model_dir.glob("*.pkl"))

    if len(model_files) != 1:
        raise RuntimeError(
            f"{model_dir} should contain exactly one model checkpoint; "
            f"found {len(model_files)}: {[p.name for p in model_files]}"
        )

    model_path = model_files[0]

    subprocess.run(
        [
            sys.executable,
            "bin/TCM_embedding_generator.py",
            "--model-dir",
            str(model_path),
            "--formula-data",
            "data/TCM_formulas/TCM_formula_data_example_2000.pkl",
            "--TCM-out-dir",
            f"results/embeddings/TCM_embeddings_repeated/{repeat_name}",
        ],
        check=True,
    )

### 3.3 Interactive example: represent and decode a symptom pattern or formula

The trained model supports bidirectional symptom-pattern/formula inference. The examples below are intended only to demonstrate the model interface.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

generator = TCMEmbeddingGenerator(
    model_dir="core/trained_model/model_epoch_60.pkl",
    device=device,
)

#### Symptom pattern → TCM-ES embedding and candidate herb generation

In [ ]:
example_symptom_pattern = ["消瘦", "食少", "气短", "大便溏薄", "神疲"]

symptom_pattern_embedding = generator.generate_symptom_embedding(
    [example_symptom_pattern]
)

symptom_to_herb_decoding = generator.herb_decoding(
    example_symptom_pattern,
    printout=True,
)

print("Symptom-pattern embedding shape:", symptom_pattern_embedding.shape)

#### Formula → TCM-ES embedding and associated symptom generation

In [ ]:
example_formula = ["射干", "细辛", "山药", "枳实", "橘皮", "藿香"]

formula_embedding = generator.generate_herb_embedding([example_formula])

formula_to_symptom_decoding = generator.symptom_decoding(
    example_formula,
    printout=True,
)

print("Formula embedding shape:", formula_embedding.shape)

### 3.4 Prepare co-occurrence-based comparison embeddings

The manuscript compares the learned TCM-ES with co-occurrence-based representations. These baselines are trained independently of the TCM-ES model.

In [ ]:
subprocess.run(
    [
        sys.executable,
        "bin/prepare_embedding_baselines.py",
        "--cooc-weighting",
        "ppmi",
    ],
    check=True,
)

## 4. Characterize TCM-ES geometry and model attention

This section examines whether the learned representation has reproducible geometric structure and whether its organization is associated with conventional TCM principles that were not used to train the core model.

### 4.1 Attention-guided token-removal perturbation

In [ ]:
subprocess.run(
    [
        sys.executable,
        "bin/attention_perturbation_analysis.py",
        "--random-repeats",
        "10",
        "--bootstrap-repeats",
        "1000",
        "--seed",
        "42",
    ],
    check=True,
)

The perturbation analysis tests whether tokens assigned higher directional cross-attention weights are more important for opposite-modality reconstruction than randomly selected or low-attention tokens. This evaluates predictive relevance of the learned attention ranking and should not be interpreted as causal or biological-mechanistic evidence.

### 4.2 Identify principal directions of TCM-ES using PCA

In [ ]:
source_name = "original"
embedding_dir = "results/embeddings/TCM_embeddings"
pca_embedding_type = "average"

out_dir = f"results/TCM_embedding_analysis/{source_name}({pca_embedding_type})"
pca_model = Path(out_dir) / "pca" / "TCM_embedding_pca_model.pkl"

subprocess.run(
    [
        sys.executable,
        "bin/TCM_embedding_pca_projection.py",
        "--embedding-dir",
        embedding_dir,
        "--out-dir",
        out_dir,
        "--pca-components",
        "6",
        "--pca-embedding-type",
        pca_embedding_type,
        # To reuse an existing PCA model:
        # "--pca-model", str(pca_model),
    ],
    check=True,
)

### 4.3 Evaluate organization associated with conventional TCM principles

In [ ]:
source_name = "original"
pca_embedding_type = "average"
analysis_dir = f"results/TCM_embedding_analysis/{source_name}({pca_embedding_type})"

subprocess.run(
    [
        sys.executable,
        "bin/TCM_principle_alignment_analysis.py",
        "--analysis-dir",
        analysis_dir,
        "--plot",
        "syndrome",
        "--embedding-side",
        "both",
    ],
    check=True,
)

The conventional TCM annotations evaluated here are used for downstream characterization of TCM-ES and were not supplied as training targets for the core model.

## 5. Reproducibility across independent training runs

Ten TCM-ES models were independently trained using different splits and random initializations. The analyses below evaluate global pairwise geometry, local nearest-neighbour consistency, and correspondence among independently fitted principal components.

For manuscript-scale reproducibility analyses, use embeddings generated from the complete model-eligible formula set rather than the small public demonstration subset.

### 5.1 Global and local embedding-geometry reproducibility

In [ ]:
subprocess.run(
    [
        sys.executable,
        "bin/embedding_robustness_evaluation.py",
        "--mode",
        "all",
        "--repeat-model-root",
        "core/trained_model_repeated",
        "--primary-pca-analysis-dir",
        "results/TCM_embedding_analysis/original(average)",
    ],
    check=True,
)

### 5.2 Align independently fitted principal components

In [ ]:
subprocess.run(
    [
        sys.executable,
        "bin/TCM_embedding_pca_repeat_alignment.py",
        "--main-pca",
        "results/TCM_embedding_analysis/original(average)/pca/TCM_embedding_pca_model.pkl",
        "--main-embedding-dir",
        "results/embeddings/TCM_embeddings",
        "--repeat-embedding-root",
        "results/embeddings/TCM_embeddings_repeated",
        "--out-dir",
        "results/TCM_embedding_pca_repeat_alignment",
        "--n-components",
        "6",
    ],
    check=True,
)

## 6. Formula prediction and retrospective clinical analyses

These analyses examine whether TCM-ES relationships are consistent with held-out formula prediction, an external TCM-PD benchmark, and observed retrospective clinical relationships.

The public clinical datasets in this repository are demonstration subsets. Their numerical estimates should not be interpreted as reproductions of the full-cohort manuscript results.

### 6.1 Internal held-out formula prediction

In [ ]:
subprocess.run(
    [
        sys.executable,
        "bin/formula_prediction_internal.py",
        "--formula-data",
        "data/TCM_formulas/TCM_formula_data_example_2000.pkl",
        "--embedding-dir",
        "results/embeddings/TCM_embeddings",
        "--train-idx",
        "data/training/train_idx.pkl",
        "--val-idx",
        "data/training/val_idx.pkl",
        "--test-idx",
        "data/training/test_idx.pkl",
        "--out-dir",
        "results/TCM_formula_prediction_internal",
        "--hidden1",
        "1024",
        "--hidden2",
        "512",
        "--dropout",
        "0.2",
        "--seed",
        "42",
    ],
    check=True,
)

### 6.2 External formula prediction on the TCM-PD benchmark

In [ ]:
subprocess.run(
    [
        sys.executable,
        "bin/formula_prediction_external.py",
        "--external-formula-data",
        "data/TCM_PD_external/TCM_PD_formula_data_external_all_valid.pkl",
        "--external-embedding-dir",
        "results/embeddings/TCM_PD_external/original",
        "--external-split-dir",
        "results/TCM_formula_prediction_external/data_split",
        "--out-dir",
        "results/TCM_formula_prediction_external",
        "--reference-formula-data",
        "data/TCM_formulas/TCM_formula_data_example_2000.pkl",
        "--external-val-ratio",
        "0.1",
        "--external-symptom-jaccard",
        "0.8",
        "--external-herb-jaccard",
        "0.8",
        "--k-values",
        "5,10,20",
        "--ridge-lambda",
        "10",
        "--hidden1",
        "1024",
        "--hidden2",
        "512",
        "--dropout",
        "0.2",
        "--seed",
        "42",
    ],
    check=True,
)

### 6.3 General TCM clinical cases

This analysis evaluates within-case relationships between prescribed formulas and subsequently alleviated versus unalleviated symptoms, together with symptom-pattern/formula retrieval. Downstream analysis uses the model-eligible clinical dataframe so that all formulas are representable by the trained TCM-ES model.

In [ ]:
subprocess.run(
    [
        sys.executable,
        "bin/clinical_distance_based_analysis.py",
        "--clinical-data",
        "data/general_TCM_clinical_cases/TCM_general_cases_data_example_500_model_eligible.pkl",
        "--symptom-list",
        "core/standard_TCM_entities/symptom_list.pkl",
        "--tcm-clinical-emb-dir",
        "results/embeddings/general_TCM_clinical_cases/original",
        "--tcm-emb-dir",
        "results/embeddings/TCM_embeddings",
        "--cooc-svd-emb-dir",
        "results/embedding_baseline_comparison/baseline_ppmi_svd",
        "--cooc-graph-emb-dir",
        "results/embedding_baseline_comparison/baseline_ppmi_graph",
        "--out-dir",
        "results/clinical_distance_based_analysis",
        "--set-metric",
        "euclidean",
        "--n-candidates",
        "50",
        "--n-repeats",
        "100",
        "--seed",
        "42",
    ],
    check=True,
)

### 6.4 COVID-19 condition–formula relationships and subsequent improvement

COVID-19 uses a study-specific standardized symptom representation. The first step generates initial-condition, symptom-change, and prescribed-formula embeddings; the second evaluates the association between condition–formula proximity and subsequent improvement.

In [ ]:
subprocess.run(
    [
        sys.executable,
        "bin/COVID_19_embedding_generator.py",
        "--data-dir",
        "data/COVID_19_data",
        "--covid-data-file",
        "COVID_data_example_200.pkl",
        "--clinical-out",
        "data/COVID_19_data/COVID_19_data_for_embeddings_example_200.pkl",
        "--out-dir",
        "results/embeddings/COVID_19_cases/example_200",
    ],
    check=True,
)

In [ ]:
subprocess.run(
    [
        sys.executable,
        "bin/COVID_condition_improvement_analysis.py",
        "--clinical-data",
        "data/COVID_19_data/COVID_19_data_for_embeddings_example_200.pkl",
        "--raw-covid-csv",
        "data/COVID_19_data/COVID_data_example_200.csv",
        "--covid-symptom-list",
        "data/COVID_19_data/COVID_symptom_list.pkl",
        "--tcm-covid-emb-dir",
        "results/embeddings/COVID_19_cases/example_200",
        "--include-formulas",
        "方D,方4,方B,方A",
        "--min-followup-days",
        "0",
        "--max-followup-days",
        "100",
        "--min-initial-symptoms",
        "1",
        "--min-initial-total-score",
        "0",
        "--max-initial-total-score",
        "100",
        "--improvement-outcome",
        "total_improvement_score",
        "--tcm-distance-metric",
        "euclidean",
        "--adjusted-min-n",
        "20",
        "--sensitivity-min-n",
        "10",
        "--adjusted-n-bins",
        "10",
        "--n-bootstrap",
        "1000",
        "--seed",
        "42",
        "--run-sensitivity-analyses",
        "--out-dir",
        "results/COVID_19_condition_improvement",
    ],
    check=True,
)

## 7. Map biomedical entities into the fixed TCM-ES

Biomedical information is introduced **after** construction of the core TCM-ES. The core embedding space remains fixed while biomedical entities are mapped through established relationships with TCM entities.

- Diseases are represented through associated symptom patterns.
- Target proteins are aligned through documented herb–target associations.
- Herbal compounds are represented using chemical-structure vectors and aligned through documented herb–compound associations.

These mappings support downstream biological characterization and exploratory hypothesis generation; embedding proximity should not be interpreted as validated therapeutic or mechanistic evidence.

### 7.1 Train target-protein alignment models and generate target embeddings

#### Primary TCM-ES

In [ ]:
subprocess.run(
    [
        sys.executable,
        "bin/train_protein_embedding.py",
        "--embedding-dir",
        "results/embeddings/TCM_embeddings",
        "--output-dir",
        "results/protein_alignment_embeddings/main",
    ],
    check=True,
)

#### Ten independently trained TCM-ES models

In [ ]:
for repeat_id in range(1, 11):
    repeat_name = f"repeat_{repeat_id:02d}"

    subprocess.run(
        [
            sys.executable,
            "bin/train_protein_embedding.py",
            "--embedding-dir",
            f"results/embeddings/TCM_embeddings_repeated/{repeat_name}/",
            "--output-dir",
            f"results/protein_alignment_embeddings/{repeat_name}/",
        ],
        check=True,
    )

### 7.2 Herbal-compound structural preprocessing and alignment

Herbal compounds are represented from SMILES using the pretrained Mol2Vec featurizer. In the current environment, this preprocessing produces 300-dimensional structural vectors. The compound-alignment model detects the actual feature dimension automatically and maps the structural representation into the 256-dimensional TCM-ES through reconstruction and compound–herb contrastive alignment.

#### Generate Mol2Vec structural features

In [ ]:
subprocess.run(
    [
        sys.executable,
        "bin/prepare_compound_mol2vec_features.py",
        "--compound-list",
        "data/herb_compounds/compound_list.pkl",
        "--compound-table",
        "data/herb_compounds/compound_SMILES.xlsx",
        "--sheet-name",
        "filtered",
        "--output-file",
        "data/herb_compounds/compound_mol2vec_vectors.pkl",
    ],
    check=True,
)

#### Train the compound-alignment model and generate compound embeddings

In [ ]:
subprocess.run(
    [
        sys.executable,
        "bin/train_compound_embedding.py",
        "--embedding-dir",
        "results/embeddings/TCM_embeddings",
        "--output-dir",
        "results/compound_alignment_embeddings/main",
    ],
    check=True,
)

### 7.3 Examine concordance between integrated TCM-ES geometry and the human PPI network

The PPI analysis evaluates whether embedding relationships among mapped entities are concordant with established molecular-network relationships. The repeat-model analysis summarizes identical pairwise relationships across independently trained TCM-ES models.

In [ ]:
subprocess.run(
    [
        sys.executable,
        "bin/PPI_concordance_analysis.py",
        "--seed",
        "42",
        "--consensus-method",
        "median",
        "--output-dir",
        "results/PPI_concordance_analysis",
    ],
    check=True,
)

## End of demonstration workflow

For a manuscript-scale run, replace the public example datasets with the corresponding full datasets where access is permitted, while preserving the same file schemas and analysis interfaces.

For command-specific options, run:

```bash
python bin/<script_name>.py --help
```

The public example data are intended to demonstrate the workflow. Their generated statistics and figures should not be interpreted as exact reproductions of the manuscript results.